In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
print(sys.executable)

/home/mxmlabserver/jpk-work/ReviewerNeuralComputing/.venv/bin/python


In [3]:
import torch
import numpy as np

from world_model_runner import WorldModelRunner
from tm_tape import TMTape
from tm_agent import TMAgent, TMAgentBrain
from tm_reviewer import TMReviewer
import tm_tasks
from reviewer_training import train

In [4]:
tasks = tm_tasks.generate_contains_one_tasks(400)

In [5]:
test_tm_agent = TMAgent(TMAgentBrain(2, 8, 8, 3), 8)
print(sum(p.numel() for p in test_tm_agent.brain.parameters()))

403


In [6]:
reviewer = TMReviewer(TMAgentBrain(2, 8))
print(sum(p.numel() for p in reviewer.parameters()))

for p in reviewer.parameters():
    print(p ,"->", p.numel())

1903919
Parameter containing:
tensor([1., 1., 1.,  ..., 1., 1., 1.], requires_grad=True) -> 1175
Parameter containing:
tensor([0., 0., 0.,  ..., 0., 0., 0.], requires_grad=True) -> 1175
Parameter containing:
tensor([[ 0.0035, -0.0016,  0.0167,  ...,  0.0097,  0.0235,  0.0008],
        [-0.0044, -0.0169,  0.0071,  ..., -0.0189, -0.0023,  0.0288],
        [-0.0074,  0.0245,  0.0258,  ..., -0.0104, -0.0070,  0.0017],
        ...,
        [ 0.0232, -0.0222,  0.0046,  ..., -0.0258,  0.0044, -0.0248],
        [ 0.0167,  0.0075,  0.0222,  ...,  0.0207, -0.0107,  0.0005],
        [ 0.0079, -0.0015,  0.0059,  ...,  0.0180, -0.0276, -0.0057]],
       requires_grad=True) -> 1203200
Parameter containing:
tensor([-0.0149,  0.0259,  0.0013,  ...,  0.0170,  0.0140,  0.0131],
       requires_grad=True) -> 1024
Parameter containing:
tensor([[-0.0111, -0.0144, -0.0305,  ...,  0.0278,  0.0255,  0.0186],
        [ 0.0237, -0.0213, -0.0204,  ...,  0.0265, -0.0097,  0.0115],
        [ 0.0291,  0.0269, -0.03

In [ ]:
training_samples = tm_tasks.generate_reviewer_training_data(tasks, 2_000_000, 2, 8)

Started generating 2000000 training samples...
Prepared 400 tasks
Generated agents 10000/2000000
Generated agents 20000/2000000
Generated agents 30000/2000000
Generated agents 40000/2000000
Generated agents 50000/2000000
Generated agents 60000/2000000
Generated agents 70000/2000000
Generated agents 80000/2000000
Generated agents 90000/2000000
Generated agents 100000/2000000
Generated agents 110000/2000000
Generated agents 120000/2000000
Generated agents 130000/2000000
Generated agents 140000/2000000
Generated agents 150000/2000000
Generated agents 160000/2000000
Generated agents 170000/2000000
Generated agents 180000/2000000
Generated agents 190000/2000000
Generated agents 200000/2000000
Generated agents 210000/2000000
Generated agents 220000/2000000
Generated agents 230000/2000000
Generated agents 240000/2000000
Generated agents 250000/2000000
Generated agents 260000/2000000
Generated agents 270000/2000000
Generated agents 280000/2000000
Generated agents 290000/2000000
Generated agent

In [9]:
print(reviewer.params_dim)

1175


In [16]:
train(reviewer, training_samples, log_every=1)

training on data set with n = 1000


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x128 and 64x1)

In [ ]:
test_tasks = tm_tasks.generate_contains_one_tasks(50)

test_samples = tm_tasks.generate_reviewer_training_data(tasks, 50, 2, 8)

/home/mxmlabserver/jpk-work/ReviewerNeuralComputing/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py:1778: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [ ]:
criterion = torch.nn.BCELoss()

correct = 0
for tape, agent, answer in test_samples:
    output = reviewer(tape, agent)

    prediction = float(output) > 0.5

    if prediction:
        print(f"Predicted: True, Answer: {answer}")

    if prediction == (answer == 1):
        correct += 1

print(f"Correct: {correct}, Ration: {float(correct)/float(len(test_samples))}")

Correct: 7837, Ration: 0.7837
